# TP 2 - Assistant RAG amélioré

---
## 0. Configuration partagée


Ce notebook compare trois stratégies de recherche avancées sur la base V2.
La requête utilisateur et le prompt système restent identiques: seule la méthode de recherche change.
Concrètement, cela permet de comparer les résultats sans biais.

In [1]:
import json

from shared.config import ROOT_DIR
from shared.llm_utils import LLMRequest, run_llm
from shared.rag_utils import (
    RAGAssistant,
    RAGChunk,
    rag_deduplicate_and_sort_chunks,
)

DATA_DIR = ROOT_DIR / "TP2_travel_planner_RAG" / "data"
VECTOR_DB_DIR = DATA_DIR / "chroma_db_rag_v3" # choisir entre chroma_db_rag_v1 et chroma_db_rag_v2 selon la base à tester

rag_assistant = RAGAssistant(persist_dir=VECTOR_DB_DIR)

MULTI_QUERY_COUNT = 3
MULTI_QUERY_TOP_K = 10
HYDE_TOP_K = 10
RERANK_FINAL_K = 10

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


---
## 1. Entrées partagées


La requête et le prompt système sont communs aux 3 méthodes.
Ne pas les modifier entre sections: sinon la comparaison devient invalide.

In [2]:
user_query = """
Je veux partir 4 jours à Rome en avril, je n'ai pas encore les dates exactes.
Propose-moi un itinéraire. Mon budget est de 200 euros pour les sorties et les restaurants.
Je veux éviter les zones trop touristiques et découvrir des lieux plus confidentiels.
"""

system_prompt = """
Tu es un assistant de planification de voyage factuel basé sur la méthode RAG.

### Règles:
- Utiliser uniquement les faits présents dans CONTEXTE.
- Ne jamais inventer prix, dates, horaires, adresses ou transports.
- Si une information manque, écrire: "Je ne sais pas à partir du contexte fourni."
- Citer chaque affirmation factuelle au format [source - chunk id].

### Contraintes de style:
- Être concis mais précis.
- Pas de mise en forme Markdown décorative.
- Pour chaque recommandation: 1 raison courte + 1 détail pratique (horaire, lieu, budget, logistique).
- Préférer des suggestions concrètes aux phrases vagues.

### Format de réponse:
1) Résumé: réponse courte (2 à 4 phrases)
2) Plan: séquence pratique adaptée à la demande
3) Informations manquantes: liste concise des faits absents
"""

---
## 2. Recherche Multi-Query


**TODO — Recherche Multi-Query**

Fichier à modifier : `TP2_travel_planner_RAG/2_4_rag_assistant_improved.ipynb`

Une seule formulation de question peut manquer des passages pertinents si les mots diffèrent de ceux des documents

Bloc de code qui génère plusieurs requêtes proches de la demande utilisateur, lance la recherche RAG pour chacune, puis fusionne les chunks récupérés

Étapes :
1. Envoyer la requête utilisateur au LLM avec `multi_query_system_prompt` pour générer `MULTI_QUERY_COUNT` requêtes distinctes
2. Pour chaque requête générée, appeler `rag_assistant.search()` avec `top_k=MULTI_QUERY_TOP_K`
3. Fusionner tous les résultats et supprimer les doublons avec `rag_deduplicate_and_sort_chunks()`
4. Conserver les `RERANK_FINAL_K` meilleurs chunks

Contrôle à faire dans la cellule d'inspection : vérifier si les requêtes générées sont réellement différentes, et pas seulement des paraphrases

Pourquoi c'est utile : une seule formulation peut rater des passages pertinents. Plusieurs formulations augmentent la couverture


In [3]:
# TODO : générer des requêtes complémentaires non redondantes
multi_query_system_prompt = f"""
Tu réécris des requêtes pour la recherche RAG.
Génère des requêtes variées qui conservent exactement l'intention utilisateur.
Règles :

(1) Interpréter la demande utilisateur
- Prendre en compte les préférences (ex: peu touristique, végétarien, etc.)
- Retirer les contraintes qui seront traitées par la récupération de chunks (ex: dates)

(2) Générer des requêtes complémentaires
- Utiliser des reformulations et synonymes réellement différents
- Formuler chaque requête comme du texte qu'on peut retrouver dans des chunks de guide

(3) Format de sortie
- Ne pas répondre à la question
- Retourner uniquement un JSON avec le schéma:
    {{\"queries\": [{', '.join(f'\"q{i}\"' for i in range(1, MULTI_QUERY_COUNT + 1))}]}}
"""

multi_query_user_prompt = (
    f"Demande utilisateur :\n{user_query}\n\n"
)
 
multi_query_result = await run_llm(
    LLMRequest(
        system_prompt=multi_query_system_prompt,
        user_prompt=multi_query_user_prompt
    )
)


INFO:google_genai.models:AFC is enabled with max remote calls: 10.


### Inspection: requêtes générées

Vérifier que les reformulations sont vraiment différentes.
Concrètement, chaque requête doit apporter un vocabulaire ou un angle nouveau; sinon elle coûte des appels API sans gain.

In [4]:
multi_query_raw_json = (
    multi_query_result.output.strip()
    .removeprefix("```json")
    .removeprefix("```")
    .removesuffix("```")
    .strip()
)
generated_queries = json.loads(multi_query_raw_json)["queries"]

print(f"Requête utilisateur d'origine :\n  {user_query.strip()}\n")
print(f"{len(generated_queries)} requêtes de recherche générées :")
for i, q in enumerate(generated_queries, start=1):
    print(f"  Q{i}: {q}")

Requête utilisateur d'origine :
  Je veux partir 4 jours à Rome en avril, je n'ai pas encore les dates exactes.
Propose-moi un itinéraire. Mon budget est de 200 euros pour les sorties et les restaurants.
Je veux éviter les zones trop touristiques et découvrir des lieux plus confidentiels.

3 requêtes de recherche générées :
  Q1: Itinéraire de 4 jours à Rome pour découvrir des lieux confidentiels, loin des foules touristiques, avec un budget de 200€ pour les sorties et restaurants.
  Q2: Explorer les trésors cachés de Rome et vivre une expérience authentique pendant 4 jours, en évitant les zones les plus fréquentées et en respectant un budget de 200€ pour les activités et repas.
  Q3: Suggestions d'activités et d'adresses locales pour un séjour de 4 jours à Rome, axé sur l'authenticité et les sites méconnus, avec un budget de 200€ pour les loisirs et la restauration.


### Récupérer et dédupliquer les chunks

Chaque requête est envoyée séparément au vecteur store.
Ensuite, fusionner les résultats et retirer les doublons pour ne garder qu'une occurrence par chunk.

In [7]:
multi_query_chunks: list[tuple[RAGChunk, float]] = []
for generated_query in generated_queries:
    multi_query_chunks.extend(rag_assistant.Research(query=str(generated_query), top_k=MULTI_QUERY_TOP_K))


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
INFO:shared.rag_utils:Embeddings calculés avec succès pour 1 textes
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
INFO:shared.rag_utils:Embeddings calculés avec succès pour 1 textes
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
INFO:shared.rag_utils:Embeddings calculés avec succès pour 1 textes


**TODO** : Finalement, retirer les doublons et trier par score vectoriel

In [8]:
multi_query_final_chunks = rag_deduplicate_and_sort_chunks(multi_query_chunks)[:RERANK_FINAL_K]

### Inspection des chunks récupérés

Contrôle concret: vérifier manuellement que les chunks retenus répondent bien à la demande.

In [9]:
print(f"Total de chunks récupérés : {len(multi_query_chunks)}")
print(f"Après déduplication     : {len(multi_query_final_chunks)}")
print(f"Fichiers et volume :")
source_counts = {}
for chunk, _score in multi_query_final_chunks:
    source = chunk.source
    if source in source_counts:
        source_counts[source] += 1
    else:
        source_counts[source] = 1
for source, count in source_counts.items():
    print(f"  {source}: {count} chunks")
print("\n"*5)

for rank, (chunk, score) in enumerate(multi_query_final_chunks, start=1):
    print(f"#{rank} | score={score:.4f} | source={chunk.source} | chunk_id={chunk.chunk_id}")
    print(chunk.text)
    print("\n"*5)

Total de chunks récupérés : 30
Après déduplication     : 10
Fichiers et volume :
  rome_5_days_guide.md: 9 chunks
  rome_restaurants.md: 1 chunks






#1 | score=0.5973 | source=rome_5_days_guide.md | chunk_id=0
Document: Rome 5 Days Guide
Section: 5-day Rome City Guide
Sous-section: Jour 1
Paragraphe: Programme

||**LEAVE HOTEL**|**LEAVE HOTEL**|Tested|and recommended hotels in Rome >|and recommended hotels in Rome >||
|---|---|---|---|---|---|---|
||Take Metro line B to Colosseo station||||||
|09:00-10:30|**Colosseum**||||Iconic symbol of|Page 5|
||||||Imperial Rome||
||Take a walk to Arch of Constantine - 5'||||||
|10:35-10:45|**Arch of Constantine**||||Majestic monument|Page 5|
||Take a walk to Roman Forum|||- 5'|||
|10:50-13:20|**Roman Forum and Palatine Hill**||||Center of the ancient|Page 6|
||||||world||
||Lunch time||||||
||Take a walk to Piazza Venezia||||||
|15:30-15:50|**Piazza Venezia**||||Focal point of modern|Page 6|
||||||Rome||
|15:50-16:20|**Vittorio Emanuele II Monu

### Générer la réponse ancrée

In [10]:
multi_query_context = "\n\n".join(
    [f"[{chunk.source} - chunk {chunk.chunk_id}]\n{chunk.text}" for chunk, _score in multi_query_final_chunks]
)
multi_query_grounded_system_prompt = f"{system_prompt}\n\nCONTEXTE:\n{multi_query_context}"

multi_query_run_result = await run_llm(
    LLMRequest(
        system_prompt=multi_query_grounded_system_prompt,
        user_prompt=user_query
    )
)
multi_query_answer = multi_query_run_result.output

print("Réponse Multi-Query")
print("------------------")
print(multi_query_answer)

INFO:google_genai.models:AFC is enabled with max remote calls: 10.


Réponse Multi-Query
------------------
Voici une proposition d'itinéraire de 4 jours à Rome, axée sur des découvertes moins conventionnelles et respectant votre budget.

1) Résumé:
Cet itinéraire de 4 jours à Rome vous propose d'explorer des quartiers authentiques et des sites historiques moins fréquentés. Vous découvrirez le charme du quartier juif, l'atmosphère bohème de Trastevere, l'histoire d'Ostia Antica et la splendeur des jardins de Tivoli, tout en gardant un œil sur votre budget.

2) Plan:

Jour 1 : Quartier Juif et Trastevere
Matin : Explorez le quartier juif, en commençant par la Piazza Costaguti [chunk 16]. Promenez-vous dans ses rues chargées d'histoire, découvrez la synagogue et le musée juif [chunk 16].
Après-midi : Dirigez-vous vers le quartier de Trastevere [chunk 23]. Flânez dans ses ruelles étroites et ses places animées, et visitez la Basilique di Santa Maria in Trastevere [chunk 17].

Jour 2 : Ostia Antica
Journée : Prenez le train pour Ostia Antica [chunk 29]. Vis

---
## 3. Recherche HyDE


**TODO — Recherche HyDE**

Fichier à modifier : `TP2_travel_planner_RAG/2_4_rag_assistant_improved.ipynb`

La question utilisateur et les passages de réponse n'utilisent pas toujours le même vocabulaire

Bloc de code qui crée un texte hypothétique avec le LLM, utilise ce texte comme requête de recherche, puis construit la réponse finale à partir des chunks trouvés

Étapes :
1. Demander au LLM d'écrire un passage court et factuel qui répondrait à la requête utilisateur, avec `hyde_system_prompt`
2. Utiliser ce texte généré (et non la question brute) comme entrée de `rag_assistant.search()`
3. Générer la réponse finale à partir des chunks récupérés

Contrôle à faire dans la cellule d'inspection : le texte hypothétique doit ressembler à un extrait de guide, dense et factuel. Un texte vague donne de mauvais résultats

Pourquoi c'est utile : le vocabulaire d'une question et celui d'un passage de guide peuvent être différents. HyDE réduit cet écart


In [11]:
# TODO : produire un texte HyDE factuel et riche en entités
hyde_system_prompt = """
Rédige un passage hypothétique pour la recherche HyDE.
Style: neutre, factuel, dense en information.
Inclure les entités/termes/synonymes probables de la requête pour améliorer la recherche.
Ne pas ajouter de méta-commentaire, de puces ni de JSON.
"""

hyde_result = await run_llm(LLMRequest(system_prompt=hyde_system_prompt, user_prompt=user_query))
hyde_text = hyde_result.output

INFO:google_genai.models:AFC is enabled with max remote calls: 10.


### Inspection: document hypothétique

Ce texte sert de requête de recherche.
Vérifier qu'il ressemble à une vraie page de guide: lieux précis, détails concrets, style informatif.

In [12]:
print("Document hypothétique généré par le LLM :")
print("=" * 60)
print(hyde_text)
print("=" * 60)
print(f"\nLongueur : {len(hyde_text)} caractères")

Document hypothétique généré par le LLM :
Un programme de voyage hypothétique pour un séjour de quatre jours à Rome en avril est esquissé, ciblant une enveloppe budgétaire de 200 euros pour les dépenses sur place, incluant les sorties et les repas. L'objectif est de privilégier la découverte de sites et de quartiers moins fréquentés, s'éloignant ainsi des zones touristiques traditionnelles et surpeuplées. Cet itinéraire potentiel mettrait l'accent sur l'exploration d'espaces confidentiels de la capitale italienne, tels que des quartiers émergents, des marchés locaux authentiques, des églises moins connues mais riches en histoire, et des parcs urbains offrant des perspectives originales sur la ville. Les suggestions gastronomiques se concentreraient sur des trattorias familiales et des osterias proposant une cuisine romaine traditionnelle à des prix abordables, permettant une immersion dans la vie quotidienne des habitants et une expérience culturelle plus profonde, tout en optimisant l

### Récupérer les chunks avec le document hypothétique

In [13]:
hyde_chunks = rag_assistant.Research(query=hyde_text, top_k=HYDE_TOP_K)

INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
INFO:shared.rag_utils:Embeddings calculés avec succès pour 1 textes


In [14]:
print(f"Total de chunks récupérés : {len(hyde_chunks)}")
print(f"Fichiers et volume :")
source_counts = {}
for chunk, _score in hyde_chunks:
    source = chunk.source
    if source in source_counts:
        source_counts[source] += 1
    else:
        source_counts[source] = 1
for source, count in source_counts.items():
    print(f"  {source}: {count} chunks")
print("\n"*5)

for rank, (chunk, score) in enumerate(hyde_chunks, start=1):
    print(f"#{rank} | score={score:.4f} | source={chunk.source} | chunk_id={chunk.chunk_id}")
    print(f"  {chunk.text}")
    print("\n"*5)

Total de chunks récupérés : 10
Fichiers et volume :
  rome_restaurants.md: 5 chunks
  rome_5_days_guide.md: 5 chunks






#1 | score=0.6101 | source=rome_restaurants.md | chunk_id=28
  Document: Rome Restaurants
Section: Tastes and Tales of Rome — 30 Local Restaurants
Sous-section: Ouvertures récentes
Paragraphe: Eufrosino Osteria

*since 2020*

**Via di Torpignattara, 188 tel. 348 588 3932**

**Tor Pignattara**

In the beating heart of **Tor Pignattara** , among historic shops and everyday life, there's a place that seems to have stepped out of another era, but speaks straight to the present: **Eufrosino Osteria** . Its honest, no-frills approach has won over the neighbourhood with a simple yet powerful formula: **simple Italian home-style cooking** , served in a place that smells like home. The name? A tribute to **Saint Euphrosynus** , patron saint of cooks, perfectly embodying the spirit of the place: humility, passion, and a desire to nourish not just the belly, but also the soul

### Générer la réponse ancrée

In [15]:
hyde_context = "\n\n".join(
    [f"[{chunk.source} - chunk {chunk.chunk_id}]\n{chunk.text}" for chunk, _score in hyde_chunks]
)
hyde_grounded_system_prompt = f"{system_prompt}\n\nCONTEXTE:\n{hyde_context}"

hyde_run_result = await run_llm(
    LLMRequest(
        system_prompt=hyde_grounded_system_prompt,
        user_prompt=user_query
    )
)
hyde_answer = hyde_run_result.output

print("Réponse HyDE")
print("-----------")
print(hyde_answer)

INFO:google_genai.models:AFC is enabled with max remote calls: 10.


Réponse HyDE
-----------
1) Résumé:
Pour un séjour de 4 jours à Rome en avril, je peux vous proposer un itinéraire axé sur des quartiers moins fréquentés et des expériences culinaires authentiques. Le budget de 200 euros pour les sorties et restaurants est un objectif, mais les détails financiers précis ne sont pas tous disponibles dans le contexte.

2) Plan:

Jour 1: Découverte de Tor Pignattara et ses environs.
*   Raison: Explorer un quartier vivant et authentique.
*   Détail pratique: Visitez Eufrosino Osteria, un lieu qui propose une cuisine italienne simple et maison, situé dans le cœur de Tor Pignattara [rome_restaurants.md - chunk 28].

Jour 2: Exploration du quartier de Centocelle.
*   Raison: Découvrir un quartier convivial et créatif.
*   Détail pratique: Découvrez Menabò, une osteria à forme libre qui rend hommage à la cuisine méditerranéenne, située à Centocelle [rome_restaurants.md - chunk 33].

Jour 3: Immersion dans le quartier de Prati.
*   Raison: Expérimenter une cui

---
## 4. Reclassement (Reranking)


**TODO — Reranking par LLM**

Fichier à modifier : `TP2_travel_planner_RAG/2_4_rag_assistant_improved.ipynb`

Le score vectoriel trie les chunks par proximité, mais ce tri n'est pas toujours le plus pertinent pour la question métier

Bloc de code qui fusionne les candidats Multi-Query et HyDE, demande un reclassement au LLM, puis reconstruit la liste finale des chunks retenus

Étapes :
1. Fusionner `multi_query_chunks` et `hyde_chunks`, puis dédupliquer et trier avec `rag_deduplicate_and_sort_chunks()`
2. Afficher la liste des candidats classés par score vectoriel avant reranking LLM
3. Envoyer tous les candidats au LLM avec `rerank_system_prompt` et récupérer un classement JSON
4. Reclasser les candidats selon ce classement et garder les `RERANK_FINAL_K` meilleurs

Le pool de candidats combine deux approches complémentaires : Multi-Query élargit la couverture de vocabulaire, HyDE rapproche la recherche de passages de réponse

Pourquoi c'est utile : le score vectoriel seul n'est pas toujours suffisant. Le reclassement LLM améliore la pertinence finale


In [16]:
# TODO : fusionner puis reranker sans casser le mapping ID -> chunk
rerank_candidates = rag_deduplicate_and_sort_chunks(multi_query_chunks + hyde_chunks)

print(f"Chunks bruts Multi-Query : {len(multi_query_chunks)}")
print(f"Chunks HyDE              : {len(hyde_chunks)}")
print(f"Après fusion + déduplication : {len(rerank_candidates)}\n")
print("\n"*5)

print("Classement des candidats par score vectoriel (avant reranking LLM) :")
for index, (chunk, score) in enumerate(rerank_candidates, start=1):
    print(f"#{index} | score={score:.4f} | source={chunk.source} | chunk_id={chunk.chunk_id}")
    print(f"  {chunk.text}")
    print("\n"*5)
     

Chunks bruts Multi-Query : 30
Chunks HyDE              : 10
Après fusion + déduplication : 18







Classement des candidats par score vectoriel (avant reranking LLM) :
#1 | score=0.6101 | source=rome_restaurants.md | chunk_id=28
  Document: Rome Restaurants
Section: Tastes and Tales of Rome — 30 Local Restaurants
Sous-section: Ouvertures récentes
Paragraphe: Eufrosino Osteria

*since 2020*

**Via di Torpignattara, 188 tel. 348 588 3932**

**Tor Pignattara**

In the beating heart of **Tor Pignattara** , among historic shops and everyday life, there's a place that seems to have stepped out of another era, but speaks straight to the present: **Eufrosino Osteria** . Its honest, no-frills approach has won over the neighbourhood with a simple yet powerful formula: **simple Italian home-style cooking** , served in a place that smells like home. The name? A tribute to **Saint Euphrosynus** , patron saint of cooks, perfectly embodying the spirit of the place: humility, passion, and a desire t

### Reclasser les candidats avec le LLM

Le LLM note chaque candidat par rapport à la requête utilisateur.
Cette étape coûte plus cher que la recherche vectorielle, mais elle peut remonter des chunks vraiment utiles et rétrograder les faux positifs.

In [17]:
candidate_lines = []
for index, (chunk, _score) in enumerate(rerank_candidates, start=1):
    preview = chunk.text.replace("\n", " ")[:500]
    candidate_lines.append(f"ID={index} | source={chunk.source} | chunk_id={chunk.chunk_id} | text={preview}")

rerank_system_prompt = """
Tu es un module de reclassement de résultats de recherche. Note chaque passage selon sa capacité à répondre à la requête utilisateur.

À classer plus haut :
- Passages contenant des faits précis (prix, adresses, horaires, recommandations nommées)
- Passages alignés avec les contraintes utilisateur (budget, durée, préférences)

À classer plus bas :
- Sommaires, crédits éditoriaux, textes génériques
- Passages remontés par simple chevauchement lexical sans utilité réelle

Retourner uniquement un JSON: {"ranking": [id1, id2, ...]}
"""

rerank_user_prompt = (
    f"Requête utilisateur:\n{user_query}\n\n"
    f"Sélectionne exactement {RERANK_FINAL_K} chunks parmi ces candidats, du plus pertinent au moins pertinent.\n"
    "Retourne uniquement un JSON: {\"ranking\": [id1, id2, ...]}\n\n"
    + "\n".join(candidate_lines)
)

rerank_result = await run_llm(
    LLMRequest(
        system_prompt=rerank_system_prompt,
        user_prompt=rerank_user_prompt
    )
)

rerank_raw_json = (
    rerank_result.output.strip()
    .removeprefix("```json")
    .removeprefix("```")
    .removesuffix("```")
    .strip()
)
ranking = json.loads(rerank_raw_json)["ranking"]

INFO:google_genai.models:AFC is enabled with max remote calls: 10.


In [18]:
print(f"Reranking LLM : {ranking}\n")

Reranking LLM : ['4', '1', '5', '9', '15', '17', '12', '11', '3', '13']



### Inspection: résultats du reranking

Le tableau compare le nouveau rang (LLM) et l'ancien rang (vecteur).
Concrètement, ce delta montre la valeur ajoutée du reranking.

In [19]:
reranked_chunks: list[tuple[RAGChunk, float, int]] = []
for new_rank, candidate_id in enumerate(ranking, start=1):
    idx = int(candidate_id) - 1
    if idx < 0 or idx >= len(rerank_candidates):
        print(f"Avertissement: ID hors plage ignoré {candidate_id} (taille du pool: {len(rerank_candidates)})")
        continue
    chunk, score = rerank_candidates[idx]
    reranked_chunks.append((chunk, score, int(candidate_id)))

print(f"IDs retournés par le LLM : {ranking}")
print(f"Pool candidat          : {len(rerank_candidates)} chunks")
print(f"Chunks valides conservés: {len(reranked_chunks)}\n")

print(f"{'New':>3} | {'Old':>3} | {'Vector':>6} | {'Source':<30} | Preview")
print("-" * 100)
for new_rank, (chunk, score, old_rank) in enumerate(reranked_chunks, start=1):
    vec_score = score
    preview = chunk.text.replace("\n", " ")[:60]
    direction = "↑" if new_rank < old_rank else ("↓" if new_rank > old_rank else "=")
    print(f"#{new_rank:>2} | #{old_rank:>2} {direction} | {vec_score:.4f} | {chunk.source:<30} | {preview}")

IDs retournés par le LLM : ['4', '1', '5', '9', '15', '17', '12', '11', '3', '13']
Pool candidat          : 18 chunks
Chunks valides conservés: 10

New | Old | Vector | Source                         | Preview
----------------------------------------------------------------------------------------------------
# 1 | # 4 ↑ | 0.5992 | rome_restaurants.md            | Document: Rome Restaurants Section: Tastes and Tales of Rome
# 2 | # 1 ↓ | 0.6101 | rome_restaurants.md            | Document: Rome Restaurants Section: Tastes and Tales of Rome
# 3 | # 5 ↑ | 0.5987 | rome_restaurants.md            | Document: Rome Restaurants Section: Tastes and Tales of Rome
# 4 | # 9 ↑ | 0.5974 | rome_restaurants.md            | Document: Rome Restaurants Section: Tastes and Tales of Rome
# 5 | #15 ↑ | 0.5794 | rome_restaurants.md            | Document: Rome Restaurants Section: Tastes and Tales of Rome
# 6 | #17 ↑ | 0.5788 | rome_restaurants.md            | Document: Rome Restaurants Section: Tastes and T

### Générer la réponse ancrée

In [20]:

rerank_context = "\n\n".join(
    [f"[{chunk.source} - chunk {chunk.chunk_id}]\n{chunk.text}" for chunk, _score, _old_rank in reranked_chunks]
)
rerank_grounded_system_prompt = f"{system_prompt}\n\nCONTEXTE:\n{rerank_context}"

rerank_run_result = await run_llm(LLMRequest(system_prompt=rerank_grounded_system_prompt, user_prompt=user_query))
rerank_answer = rerank_run_result.output

print("Réponse après reranking")
print("----------------")
print(rerank_answer)

INFO:google_genai.models:AFC is enabled with max remote calls: 10.


Réponse après reranking
----------------
Voici une proposition d'itinéraire pour votre séjour de 4 jours à Rome en avril, axée sur la découverte de lieux confidentiels et une cuisine authentique, tout en respectant votre budget.

1) Résumé:
Cet itinéraire vous propose de découvrir Rome à travers ses quartiers moins fréquentés et ses restaurants locaux, offrant une immersion dans la vie romaine authentique. Vous explorerez des quartiers comme Tor Pignattara, Garbatella et Prati, en savourant des plats traditionnels et innovants dans des établissements conviviaux. L'accent est mis sur des expériences culinaires accessibles et des découvertes culturelles hors des sentiers battus.

2) Plan:

Jour 1 : Immersion à Tor Pignattara
Découverte du quartier de Tor Pignattara, connu pour son authenticité et sa vie de quartier [rome_restaurants.md - chunk 28].
Recommandation : Dîner à Eufrosino Osteria.
Raison : Ce restaurant propose une cuisine italienne simple et maison, dans une ambiance chaleure